# [LONG] Model Evaluation & Best Model Selection

**Mục tiêu:** Đánh giá toàn diện 5 mô hình ML, phân tích Recall theo từng lớp, và xác nhận Random Forest là mô hình tốt nhất.

**Đầu vào:** `models/model_results.csv`, `models/random_forest.pkl`, `models/all_predictions.pkl`

## [LONG] - Bước 1: Import thư viện

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

ROOT = Path('..').resolve()
PROCESSED_DIR = ROOT / 'data' / 'processed'
MODELS_DIR = ROOT / 'models'

print('Thư viện đã được tải thành công.')

## [LONG] - Bước 2: Tải dữ liệu và artifacts

In [ ]:
results_df = pd.read_csv(MODELS_DIR / 'model_results.csv')
rf_model   = joblib.load(MODELS_DIR / 'random_forest.pkl')
all_preds  = joblib.load(MODELS_DIR / 'all_predictions.pkl')
le         = joblib.load(MODELS_DIR / 'label_encoder.pkl')

y_test  = np.load(PROCESSED_DIR / 'y_test.npy')
X_test  = np.load(PROCESSED_DIR / 'X_test.npy')

target_names = le.classes_.tolist()
model_names  = results_df['Model'].tolist()

print(f'Số mô hình: {len(model_names)}')
print(f'Nhãn: {target_names}')
print(f'Shape X_test: {X_test.shape}')

## [LONG] - Bước 3: Bảng so sánh tổng hợp

In [ ]:
print('[LONG] Bảng so sánh hiệu suất 5 mô hình:')
print('=' * 75)
display_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'Train Time (s)']
print(results_df[display_cols].to_string(index=False))
print('=' * 75)

best_row = results_df.loc[results_df['F1 Score'].idxmax()]
print(f'\n→ Mô hình tốt nhất theo F1 Score: {best_row["Model"]} (F1={best_row["F1 Score"]:.4f})')

## [LONG] - Bước 4: Biểu đồ so sánh tất cả chỉ số

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
x = np.arange(len(model_names))
width = 0.18
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

fig, ax = plt.subplots(figsize=(14, 6))

for i, (metric, color) in enumerate(zip(metrics, colors)):
    offset = (i - 1.5) * width
    bars = ax.bar(x + offset, results_df[metric], width, label=metric, color=color, alpha=0.85)
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            f'{bar.get_height():.3f}',
            ha='center', va='bottom', fontsize=7, rotation=90
        )

ax.set_xlabel('Mô hình', fontsize=12)
ax.set_ylabel('Giá trị', fontsize=12)
ax.set_title('[LONG] So sánh các chỉ số hiệu suất — 5 mô hình ML', fontsize=13, pad=12)
ax.set_xticks(x)
ax.set_xticklabels(results_df['Model'], rotation=15, ha='right', fontsize=10)
ax.set_ylim(0, 1.12)
ax.legend(loc='lower right', fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(MODELS_DIR / 'metrics_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
print('Đã lưu: metrics_comparison.png')

## [LONG] - Bước 5: Ma trận nhầm lẫn cho từng mô hình

In [ ]:
n_models = len(model_names)
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, model_name in enumerate(model_names):
    y_pred = all_preds[model_name]
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
    disp.plot(ax=axes[idx], colorbar=False, xticks_rotation=45)
    axes[idx].set_title(f'[LONG] {model_name}', fontsize=11, pad=8)
    axes[idx].tick_params(axis='both', labelsize=7)

# Ẩn ô trống nếu có
for idx in range(n_models, len(axes)):
    axes[idx].set_visible(False)

fig.suptitle('[LONG] Confusion Matrix — 5 Mô hình ML', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'confusion_matrices.png', dpi=100, bbox_inches='tight')
plt.show()
print('Đã lưu: confusion_matrices.png')

## [LONG] - Bước 6: Phân tích Recall theo từng lớp tấn công

In [ ]:
from sklearn.metrics import classification_report
import json

# Thu thập Recall theo lớp cho mỗi mô hình
per_class_recall = {}

for model_name in model_names:
    y_pred = all_preds[model_name]
    report = classification_report(
        y_test, y_pred,
        target_names=target_names,
        output_dict=True,
        zero_division=0
    )
    per_class_recall[model_name] = {
        cls: report[cls]['recall']
        for cls in target_names
        if cls in report
    }

recall_df = pd.DataFrame(per_class_recall).T

# Chỉ xét các lớp TẤN CÔNG (không phải BENIGN)
attack_classes = [c for c in target_names if c != 'BENIGN']
attack_recall_df = recall_df[attack_classes]

print('[LONG] Recall theo từng lớp tấn công (không gồm BENIGN):')
print(attack_recall_df.round(4).to_string())

# Mô hình có Recall trung bình cao nhất trên các lớp tấn công
avg_attack_recall = attack_recall_df.mean(axis=1)
best_model_recall = avg_attack_recall.idxmax()
print(f'\n→ Mô hình có Recall cao nhất trên lớp tấn công: {best_model_recall}')
print(f'   (Recall trung bình các lớp tấn công = {avg_attack_recall[best_model_recall]:.4f})')

In [ ]:
# Biểu đồ heatmap Recall theo lớp
fig, ax = plt.subplots(figsize=(max(10, len(attack_classes) * 1.2), 5))
sns.heatmap(
    attack_recall_df,
    annot=True, fmt='.3f',
    cmap='RdYlGn', vmin=0, vmax=1,
    linewidths=0.4, linecolor='white',
    ax=ax, annot_kws={'size': 8}
)
ax.set_title('[LONG] Per-class Recall — Các lớp tấn công', fontsize=13, pad=12)
ax.set_xlabel('Loại tấn công', fontsize=11)
ax.set_ylabel('Mô hình', fontsize=11)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig(MODELS_DIR / 'recall_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print('Đã lưu: recall_heatmap.png')

## [LONG] - Tại sao Recall quan trọng hơn Accuracy trong IDS?

Trong bài toán phát hiện xâm nhập mạng (IDS), **Recall (Độ nhạy)** là chỉ số quan trọng hơn Accuracy vì:

### 1. Chi phí của False Negative rất cao
- **False Negative** = mô hình phân loại sai một cuộc **tấn công thực** là BENIGN
- Hậu quả: hệ thống bỏ sót cuộc tấn công → dữ liệu bị đánh cắp, hệ thống bị xâm nhập

### 2. Bộ dữ liệu mất cân bằng nghiêm trọng
- BENIGN chiếm ~80% dữ liệu → mô hình chỉ cần đoán "BENIGN" liên tục sẽ đạt Accuracy ~80%
- Nhưng Recall trên lớp tấn công sẽ = 0 — nghĩa là không phát hiện được gì!

### 3. Ưu tiên an toàn hơn tiện lợi
- **False Positive** (báo nhầm BENIGN là tấn công) chỉ gây phiền toái (kiểm tra thủ công)
- **False Negative** (bỏ sót tấn công) có thể gây thiệt hại nghiêm trọng về bảo mật

> **Kết luận:** Trong IDS, chúng ta muốn tối đa hóa Recall (không bỏ sót tấn công) dù có thể chấp nhận một số False Positive.

## [LONG] - Kết luận: Random Forest là mô hình tốt nhất

In [ ]:
rf_row = results_df[results_df['Model'] == 'Random Forest'].iloc[0]

print('=' * 60)
print('[LONG] KẾT LUẬN — Random Forest là mô hình tốt nhất')
print('=' * 60)
print(f'\n  Accuracy:  {rf_row["Accuracy"]:.4f}')
print(f'  Precision: {rf_row["Precision"]:.4f}')
print(f'  Recall:    {rf_row["Recall"]:.4f}')
print(f'  F1 Score:  {rf_row["F1 Score"]:.4f}')

print('\nLý do chọn Random Forest:')
print('  1. F1 Score và Recall cao nhất → ít bỏ sót tấn công nhất')
print('  2. Ensemble nhiều cây → ổn định, ít overfitting hơn cây đơn')
print('  3. Tự nhiên xử lý đặc trưng không cần chuẩn hóa')
print('  4. n_jobs=-1 → song song hóa trên nhiều CPU')
print('  5. Cung cấp feature_importances_ → giải thích được')
print('\n→ Random Forest được lưu tại models/random_forest.pkl')
print('   và sẽ được sử dụng trong hệ thống real-time (src/deploy.py)')

print('\n✓ [LONG] Hoàn thành Evaluation & Best Model Selection!')